# Week 2: From Raw Reviews to Batched Tensors

You will build a reusable PyTorch pipeline: **review → cleaned text → tokens → integer IDs → padded batches**.
By the end, you should be able to explain the roles of a vocabulary, `Dataset`, `collate_fn`, and `DataLoader`, and inspect a padded batch and its mask.

**Your three TODOs:** encode token lists as IDs (Step 4), retrieve one dataset example (Step 5), and pad a batch and create its mask (Step 6). The other code is supplied. Complete the TODOs before running the notebook top-to-bottom. The starter intentionally cannot finish until you supply these answers.

**Dataset correction:** the linked `Amazon_2k.csv` currently contains **5,000 reviews with ratings 1–5**, despite its filename. You will exclude neutral ratings (3), map 1–2 to negative (0) and 4–5 to positive (1), and sample exactly 2,000 reviews with a fixed seed. The resulting sample is not balanced; this exercise tests data preparation, not classification performance.

CPU is sufficient. First-time installation/download requires internet access. If you work offline, install the dependencies beforehand and place the supplied CSV beside this notebook.


## Step 1 — Check your environment
If an import fails, run `%pip install torch pandas` in a separate notebook cell, restart your kernel if prompted, and rerun from the top. No GPU or pretrained model download is required.


In [ ]:
import sys
import platform
from pathlib import Path
from collections import Counter
import re
import html
import unicodedata
from html.parser import HTMLParser
from urllib.request import urlopen

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SEED = 42
print("Python:", sys.version.split()[0], "Platform:", platform.system())
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())


## Step 2 — Load exactly 2,000 usable reviews
Upload [Amazon_2k.csv](https://github.com/DrManalJalloul/Introduction-to-NLP/blob/main/Amazon_2k.csv) beside this notebook. Both capitalized and lowercase filenames are accepted. If neither is present, the cell downloads the source CSV.

You will validate the columns and labels before sampling. The loader also accepts a prepared CSV whose labels are already 0/1. A CSV row is one review; it may contain several sentences.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/DrManalJalloul/Introduction-to-NLP/main/Amazon_2k.csv"
N_REVIEWS = 2000
paths = [Path("Amazon_2k.csv"), Path("amazon_2k.csv")]
csv_path = next((p for p in paths if p.is_file()), paths[0])
if not csv_path.is_file():
    try:
        with urlopen(DATA_URL, timeout=60) as response:
            csv_path.write_bytes(response.read())
    except Exception as exc:
        raise RuntimeError("Download failed. Upload Amazon_2k.csv beside this notebook and rerun.") from exc

raw_df = pd.read_csv(csv_path)
if not {"text", "label"}.issubset(raw_df.columns):
    raise ValueError("The CSV must contain text and label columns.")
if raw_df[["text", "label"]].isna().any().any():
    raise ValueError("Missing text or labels: inspect and resolve them before continuing.")
if not raw_df["text"].map(lambda x: isinstance(x, str) and bool(x.strip())).all():
    raise ValueError("Each review must be a nonempty string.")

df = raw_df[["text", "label"]].copy()
numeric_labels = pd.to_numeric(df["label"], errors="raise")
observed = set(numeric_labels.unique())
if observed <= {0, 1}:
    df["label"] = numeric_labels.astype(int)
    print("Source labels are already binary.")
elif observed <= {1, 2, 3, 4, 5}:
    df["rating"] = numeric_labels.astype(int)
    df = df.loc[df["rating"] != 3].copy()
    df["label"] = (df["rating"] >= 4).astype(int)
    print("Converted ratings: 1–2 → 0; 4–5 → 1; excluded rating 3.")
else:
    raise ValueError(f"Unsupported labels: {observed}")
if len(df) < N_REVIEWS:
    raise ValueError(f"Need {N_REVIEWS} usable reviews; found {len(df)}.")

df = df.sample(n=N_REVIEWS, random_state=SEED).reset_index(drop=True)
texts = df["text"].tolist()
labels = df["label"].tolist()
assert len(texts) == len(labels) == N_REVIEWS
print(f"Source rows: {len(raw_df):,}; selected reviews: {len(df):,}")
print("Selected labels (0=negative, 1=positive):", df["label"].value_counts().sort_index().to_dict())


## Step 3 — Clean and tokenize: reuse your Week 1 functions
If you have your Week 1 functions, replace the two functions below while keeping the names `clean_text` and `my_tokenizer`. Your tokenizer must return a **list of token strings**, not integer IDs. For example, a spaCy wrapper can return `[token.text for token in nlp.tokenizer(text) if not token.is_space]`.

For a self-contained run, supplied reference functions normalize Unicode, remove HTML markup, lowercase, and tokenize words and punctuation. They retain negation, numbers, punctuation, and emoji rather than remove possible sentiment cues. This simple rule-based tokenizer is a fallback, not a claim that it exactly reproduces your Week 1 tokenizer.

**Tokens and IDs are different.** A tokenizer might return `['not', 'good', '!']`; the vocabulary in the next step assigns numbers to those strings. Do not use SentencePiece IDs with this word-vocabulary path: SentencePiece requires its own vocabulary and explicit special-ID configuration.


In [ ]:
class TextOnlyParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.parts = []

    def handle_data(self, data):
        self.parts.append(data)

    def handle_starttag(self, tag, attrs):
        self.parts.append(" ")

    def handle_endtag(self, tag):
        self.parts.append(" ")

def clean_text(text):
    parser = TextOnlyParser()
    parser.feed(html.unescape(text))
    parser.close()
    cleaned = unicodedata.normalize("NFC", "".join(parser.parts)).lower()
    return re.sub(r"\s+", " ", cleaned).strip()

def my_tokenizer(text):
    return re.findall(r"\w+(?:['’]\w+)*|[^\w\s]", text, flags=re.UNICODE)

tokens = [my_tokenizer(clean_text(text)) for text in texts]
assert all(isinstance(row, list) and all(isinstance(t, str) for t in row) for row in tokens)
print("Cleaned example:", clean_text(texts[0])[:150])
print("First 15 tokens:", tokens[0][:15])


## Step 4 — Build a vocabulary and encode reviews
You reserve **0 for `<pad>`** and **1 for `<unk>`**. Actual word tokens start at 2. A token not in the vocabulary maps to `<unk>`; an empty cleaned review also becomes `[UNK_ID]` so it has a valid sequence length.

The supplied code counts tokens and builds a reproducible mapping. The frequency threshold of 2 reduces the vocabulary by mapping one-off tokens to `<unk>`.

**TODO 1:** complete `encode_tokens`. Use `word_to_id.get(token, UNK_ID)` for each token. Return a Python list of integers. The empty-list fallback is already supplied.

For this pipeline-only exercise you build the vocabulary on the selected reviews. Before later training/evaluation, split your data first and fit the vocabulary on **training reviews only**, then reuse that mapping for validation, test data, and inference.


In [ ]:
PAD_ID, UNK_ID = 0, 1
MIN_FREQ = 2
counts = Counter(token for row in tokens for token in row)
word_to_id = {"<pad>": PAD_ID, "<unk>": UNK_ID}
for token, frequency in sorted(counts.items()):
    if frequency >= MIN_FREQ and token not in word_to_id:
        word_to_id[token] = len(word_to_id)

def encode_tokens(token_list):
    # TODO 1: Look up each token; use UNK_ID for unknown words.
    ids = ...
    if not isinstance(ids, list):
        raise NotImplementedError("TODO 1: replace ... with a list of integer IDs.")
    return ids if ids else [UNK_ID]

assert encode_tokens([]) == [UNK_ID]
assert encode_tokens(["a_never_seen_test_token_98765"]) == [UNK_ID]
assert encode_tokens(["<pad>", "<unk>"]) == [PAD_ID, UNK_ID]
encoded_reviews = [encode_tokens(row) for row in tokens]
assert all(len(row) >= 1 and all(isinstance(i, int) and 0 < i < len(word_to_id) for i in row)
           for row in encoded_reviews)
print("Vocabulary size (including pad/unk):", len(word_to_id))
print("First 15 IDs:", encoded_reviews[0][:15])


## Step 5 — Retrieve one example with a Dataset
A `Dataset` stores examples and defines how to retrieve one example by index. It does not pad or assemble batches.

**TODO 2:** complete `__getitem__` by returning `{"input_ids": self.sequences[idx], "label": self.labels[idx]}`. Each example contains a variable-length list of IDs and one integer label.


In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, sequences, labels):
        if len(sequences) != len(labels):
            raise ValueError("Every review must have exactly one label.")
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        # TODO 2: Return the input_ids and label for idx in a dictionary.
        sample = ...
        if not isinstance(sample, dict):
            raise NotImplementedError("TODO 2: replace ... with the sample dictionary.")
        return sample

ds = ReviewDataset(encoded_reviews, labels)
assert len(ds) == N_REVIEWS
assert ds[0] == {"input_ids": encoded_reviews[0], "label": labels[0]}
print("Dataset length:", len(ds))
print("First example length:", len(ds[0]["input_ids"]), "Label:", ds[0]["label"])


## Step 6 — Pad a batch and construct its mask
`collate_batch` receives a list of examples and returns three tensors:

| Tensor | Shape | Type | Meaning |
|---|---|---|---|
| `ids_padded` | `[batch_size, longest_length]` | `torch.long` | Token IDs, right-padded with 0 |
| `attention_mask` | Same as IDs | `torch.bool` | `True` for real tokens, `False` for padding |
| `batch_labels` | `[batch_size]` | `torch.long` | One label per review |

**TODO 3:** pad `ids` using `pad_sequence(ids, batch_first=True, padding_value=PAD_ID)` and create the mask using `ids_padded != PAD_ID`.

The mask is called an attention mask by convention; this lab does not implement attention. **A standard `nn.LSTM` does not automatically use this mask.** In your later model, derive lengths with `attention_mask.sum(dim=1)` and use `pack_padded_sequence`, or otherwise select/mask valid time steps. Also set `padding_idx=PAD_ID` in the embedding layer; that alone does not stop the LSTM from processing padding.


In [ ]:
def collate_batch(batch):
    ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    batch_labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    # TODO 3a: Right-pad the sequences to the longest one in this batch.
    ids_padded = ...
    if not isinstance(ids_padded, torch.Tensor):
        raise NotImplementedError("TODO 3a: replace ... with pad_sequence(...).")
    # TODO 3b: True where a token is not padding; False where it is padding.
    attention_mask = ...
    if not isinstance(attention_mask, torch.Tensor):
        raise NotImplementedError("TODO 3b: replace ... with a boolean comparison.")
    return ids_padded, attention_mask, batch_labels

# A tiny test with known answers, independent of review lengths.
test_ids, test_mask, test_labels = collate_batch([
    {"input_ids": [2, 3, 4], "label": 1},
    {"input_ids": [5], "label": 0},
])
assert test_ids.tolist() == [[2, 3, 4], [5, 0, 0]]
assert test_mask.dtype == torch.bool
assert test_mask.tolist() == [[True, True, True], [True, False, False]]
assert test_labels.tolist() == [1, 0]
print("Padding and mask checks passed.")


## Step 7 — Create and validate your DataLoader
The loader shuffles examples and assembles batches of up to 32. `num_workers=0` avoids multiprocessing issues in notebooks. All tensors stay on the CPU here; move them to the model's device during training.

The checks below inspect **every batch**, including the final smaller batch, to ensure that all 2,000 examples are served and masks agree with their original sequence lengths.


In [ ]:
BATCH_SIZE = 32
loader = DataLoader(
    ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_batch, num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)

seen = 0
seen_tokens = 0
for ids_batch, mask_batch, labels_batch in loader:
    assert ids_batch.ndim == 2 and ids_batch.shape == mask_batch.shape
    assert ids_batch.dtype == labels_batch.dtype == torch.long
    assert mask_batch.dtype == torch.bool
    assert labels_batch.shape == (ids_batch.shape[0],)
    assert torch.equal(mask_batch, ids_batch != PAD_ID)
    assert (mask_batch.sum(dim=1) > 0).all().item()
    assert ((ids_batch >= 0) & (ids_batch < len(word_to_id))).all().item()
    assert ((labels_batch == 0) | (labels_batch == 1)).all().item()
    seen += labels_batch.numel()
    seen_tokens += mask_batch.sum().item()
assert seen_tokens == sum(len(row) for row in encoded_reviews)
assert seen == N_REVIEWS
assert len(loader) == 63
print(f"Validated {len(loader)} batches containing {seen:,} reviews.")


## Before you submit
Run **Kernel → Restart & Run All** (or the equivalent in your notebook environment) after completing all three TODOs. Submit your completed `.ipynb` with its outputs visible. It should end by printing a padded batch's shape and masks.

You should be able to explain why token IDs are integers, why padding must use a reserved ID, and why a mask distinguishes real tokens from added padding.

**References:** [PyTorch Dataset and DataLoader](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html), [pad_sequence](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html), [pack_padded_sequence](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html).

## Step 8 — Print one padded batch and its attention mask
Batch width depends on the longest review in this shuffled batch. A mask row has `True` at token positions and `False` at padding positions. Only the first two rows are printed to keep the output readable.


In [ ]:
batch_ids, batch_mask, batch_lbl = next(iter(loader))
print("Padded batch shape:", tuple(batch_ids.shape))
print("Attention-mask shape:", tuple(batch_mask.shape))
print("Labels shape:", tuple(batch_lbl.shape))
print("First two lengths:", batch_mask[:2].sum(dim=1).tolist())
torch.set_printoptions(threshold=10000)
print("First two padded ID rows:\n", batch_ids[:2])
print("Attention mask (first two rows; True=token, False=padding):\n", batch_mask[:2])
torch.set_printoptions(profile="default")
